# WESAD Phase 0: Preprocessing and Segmentation
This notebook converts the raw WESAD `.pkl` files into the Corponi-compatible `.h5` format, extracting segments and preserving ground truth labels for stress classification.

In [1]:
# ==============================================================================
# FASE 0 (AGGIORNATA): PREPROCESSING + SEGMENTAZIONE WESAD
# ==============================================================================
import os, sys, shutil, pickle, h5py, glob
import numpy as np
from tqdm import tqdm
import pandas as pd

# 1. Ricerca Percorsi
print(" Risoluzione percorsi WESAD...")
found_paths = glob.glob('/kaggle/input/**/S2/S2.pkl', recursive=True)
if not found_paths:
    print("ERRORE: Dataset non trovato!"); sys.exit()
WESAD_INPUT = os.path.dirname(os.path.dirname(found_paths[0]))

OUTPUT_DIR = "/kaggle/working/wesad_segmented" # Direttamente segmentato!
if os.path.exists(OUTPUT_DIR): shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Parametri di Segmentazione
SEG_LEN = 512 # Secondi
STEP = 256    # Secondi
CHANNELS = ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP']
FREQS = {'ACC_x': 32, 'ACC_y': 32, 'ACC_z': 32, 'BVP': 64, 'EDA': 4, 'TEMP': 4}

subjects = [f"S{i}" for i in range(2, 18) if i != 12]
all_seg_paths, all_seg_labels, all_seg_subs = [], [], []

for sub in tqdm(subjects, desc="Soggetti"):
    sub_path = os.path.join(WESAD_INPUT, sub, f"{sub}.pkl")
    with open(sub_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')
    
    # Estrazione segnali e allineamento (semplificato)
    wrist = data['signal']['wrist']
    labels_raw = data['label']
    
    # Calcoliamo il numero di segmenti possibili (basandoci sul BVP a 64Hz)
    n_samples_bvp = len(wrist['BVP'])
    total_sec = n_samples_bvp // 64
    n_segments = (total_sec - (SEG_LEN // 1)) // (STEP // 1) + 1
    
    if n_segments <= 0: continue
    
    sub_dir = os.path.join(OUTPUT_DIR, sub)
    os.makedirs(sub_dir, exist_ok=True)
    
    for i in range(n_segments):
        t0_sec = i * STEP
        t1_sec = t0_sec + SEG_LEN
        
        # Estrazione etichetta prevalente nel segmento
        l_start, l_end = t0_sec * 700, t1_sec * 700
        seg_label_raw = labels_raw[l_start:l_end]
        # Se c'è Stress (2), allora la label è 1, altrimenti 0
        final_label = 1.0 if (seg_label_raw == 2).any() else 0.0
        
        # Creazione file .h5 per il segmento
        seg_filename = os.path.join(sub_dir, f"{i}.h5")
        with h5py.File(seg_filename, "w") as f:
            for c in CHANNELS:
                f_rate = FREQS[c]
                if 'ACC' in c:
                    axis = 0 if '_x' in c else (1 if '_y' in c else 2)
                    sig_seg = wrist['ACC'][t0_sec*f_rate : t1_sec*f_rate, axis]
                else:
                    sig_seg = wrist[c][t0_sec*f_rate : t1_sec*f_rate].flatten()
                f.create_dataset(c, data=sig_seg)
        
        all_seg_paths.append(seg_filename)
        all_seg_labels.append([final_label]) # Formato richiesto [[l1], [l2]...]
        all_seg_subs.append(sub)

# 3. Salvataggio Metadata finale
meta = {
    "sessions_paths": np.array(all_seg_paths),
    "sessions_labels": {"status": np.array(all_seg_labels).flatten()},
    "sessions_info": {sub: {"Sub_ID": sub} for sub in subjects},
    "ds_info": {
        "segment_length": SEG_LEN,
        "channel_freq": FREQS  # FONDAMENTALE (BUG 2)
    }
}
# Aggiunta campo Sub_ID nei labels per lo split di Zaza
meta["sessions_labels"]["Sub_ID"] = np.array(all_seg_subs)

with open(os.path.join(OUTPUT_DIR, "metadata.pkl"), "wb") as f:
    pickle.dump(meta, f)

print(f"\n OK! Creati {len(all_seg_paths)} segmenti etichettati.")


 Risoluzione percorsi WESAD...


Soggetti: 100%|██████████| 15/15 [01:52<00:00,  7.50s/it]


 OK! Creati 317 segmenti etichettati.


In [2]:
# ==============================================================================
# CREAZIONE ARCHIVIO SCARICABILE (TAR.GZ)
# ==============================================================================
import os

OUTPUT_DIR = "/kaggle/working/wesad_segmented"
ARCHIVE_NAME = "/kaggle/working/wesad_segmented.tar.gz"

print(f"⚙️ Creazione dell'archivio {ARCHIVE_NAME} in corso...")
print("⏳ Potrebbe richiedere qualche minuto a seconda della dimensione...")

# Usiamo il comando !tar per massima efficienza in ambiente Linux/Kaggle
!tar -czf {ARCHIVE_NAME} -C /kaggle/working {os.path.basename(OUTPUT_DIR)}

# Verifica che il file sia stato creato
if os.path.exists(ARCHIVE_NAME):
    size_mb = os.path.getsize(ARCHIVE_NAME) / (1024 * 1024)
    print(f"✅ Archivio creato con successo! Dimensione: {size_mb:.2f} MB")
    print("👉 Ora puoi scaricarlo dal pannello di destra 'Output' di Kaggle.")
else:
    print("❌ Errore durante la creazione dell'archivio.")

⚙️ Creazione dell'archivio /kaggle/working/wesad_segmented.tar.gz in corso...
⏳ Potrebbe richiedere qualche minuto a seconda della dimensione...
✅ Archivio creato con successo! Dimensione: 37.68 MB
👉 Ora puoi scaricarlo dal pannello di destra 'Output' di Kaggle.
